# 📥 Notebook 5: Download Optimization

Make downloads fast, resumable, and efficient using range requests and parallel transfers.

## Learning Objectives

By the end of this notebook, you'll understand:
- HTTP Range requests
- Resumable downloads
- Parallel chunk downloads
- CDN basics

In [1]:
import boto3
from botocore.config import Config
import requests
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

BUCKET = 'uploads'

print("✅ Connected to MinIO!")

✅ Connected to MinIO!


## 📥 Range Requests

In [2]:
print("📥 HTTP Range Requests")
print("=" * 60)
print("""
Range requests let you download PART of a file.

NORMAL REQUEST:
─────────────────────────────────────────────────────────────
GET /large-video.mp4

Response: Entire 5GB file

─────────────────────────────────────────────────────────────

RANGE REQUEST:
─────────────────────────────────────────────────────────────
GET /large-video.mp4
Range: bytes=0-10485759

Response: Just the first 10MB (bytes 0-10485759)
Status: 206 Partial Content

USE CASES:
• Resume interrupted downloads
• Video seeking (jump to 1:30:00)
• Parallel chunk downloads
• Download just file header (metadata)
""")

📥 HTTP Range Requests

Range requests let you download PART of a file.

NORMAL REQUEST:
─────────────────────────────────────────────────────────────
GET /large-video.mp4

Response: Entire 5GB file

─────────────────────────────────────────────────────────────

RANGE REQUEST:
─────────────────────────────────────────────────────────────
GET /large-video.mp4
Range: bytes=0-10485759

Response: Just the first 10MB (bytes 0-10485759)
Status: 206 Partial Content

USE CASES:
• Resume interrupted downloads
• Video seeking (jump to 1:30:00)
• Parallel chunk downloads
• Download just file header (metadata)



In [3]:
print("📤 Creating test file for download demos...")
test_data = os.urandom(10 * 1024 * 1024)
s3.put_object(Bucket=BUCKET, Key='downloads/test-file.bin', Body=test_data)
print(f"   Created 10MB test file")

download_url = s3.generate_presigned_url(
    'get_object',
    Params={'Bucket': BUCKET, 'Key': 'downloads/test-file.bin'},
    ExpiresIn=3600
)

print("\n📥 Range Request Demo")
print("=" * 60)

print("\n1️⃣ Get file size first...")
# Presigned URLs are signed for a SPECIFIC operation (get_object here),
# so a HEAD request would return 403. Trick: ask for byte 0 only and read
# the "Content-Range: bytes 0-0/TOTAL" header to learn the full size.
probe = requests.get(download_url, headers={'Range': 'bytes=0-0'})
content_range = probe.headers.get('Content-Range', '')
total_size = int(content_range.split('/')[-1]) if '/' in content_range else 0
print(f"   Total size: {total_size / 1024 / 1024:.2f}MB")

print("\n2️⃣ Download first 1MB using Range header...")
range_response = requests.get(
    download_url,
    headers={'Range': 'bytes=0-1048575'}
)
print(f"   Status: {range_response.status_code}")
print(f"   Content-Range: {range_response.headers.get('Content-Range')}")
print(f"   Bytes received: {len(range_response.content)}")

print("\n3️⃣ Download last 1MB...")
start_byte = total_size - 1048576
range_response = requests.get(
    download_url,
    headers={'Range': f'bytes={start_byte}-{total_size-1}'}
)
print(f"   Status: {range_response.status_code}")
print(f"   Content-Range: {range_response.headers.get('Content-Range')}")
print(f"   Bytes received: {len(range_response.content)}")

📤 Creating test file for download demos...
   Created 10MB test file

📥 Range Request Demo

1️⃣ Get file size first...
   Total size: 10.00MB

2️⃣ Download first 1MB using Range header...
   Status: 206
   Content-Range: bytes 0-1048575/10485760
   Bytes received: 1048576

3️⃣ Download last 1MB...
   Status: 206
   Content-Range: bytes 9437184-10485759/10485760
   Bytes received: 1048576


## 🔄 Resumable Downloads

In [4]:
class ResumableDownloader:
    def __init__(self, url: str, chunk_size: int = 1024 * 1024):
        self.url = url
        self.chunk_size = chunk_size
        self.downloaded_bytes = 0
        self.total_bytes = 0
        self.data = bytearray()
    
    def get_total_size(self) -> int:
        # Works even for URLs signed only for GET: request byte 0 and parse
        # the total from the Content-Range response header.
        response = requests.get(self.url, headers={'Range': 'bytes=0-0'})
        content_range = response.headers.get('Content-Range', '')
        if '/' in content_range:
            self.total_bytes = int(content_range.split('/')[-1])
        else:
            self.total_bytes = int(response.headers.get('Content-Length', 0))
        return self.total_bytes
    
    def download_chunk(self, start: int, end: int) -> bytes:
        response = requests.get(
            self.url,
            headers={'Range': f'bytes={start}-{end}'}
        )
        return response.content
    
    def download(self, simulate_failure_at: float = None):
        if self.total_bytes == 0:
            self.get_total_size()
        
        self.data = bytearray()
        self.downloaded_bytes = 0
        
        while self.downloaded_bytes < self.total_bytes:
            start = self.downloaded_bytes
            end = min(start + self.chunk_size - 1, self.total_bytes - 1)
            
            if simulate_failure_at and self.downloaded_bytes / self.total_bytes >= simulate_failure_at:
                raise ConnectionError(f"Simulated failure at {simulate_failure_at*100:.0f}%")
            
            chunk = self.download_chunk(start, end)
            self.data.extend(chunk)
            self.downloaded_bytes += len(chunk)
            
            progress = self.downloaded_bytes / self.total_bytes * 100
            yield progress
    
    def resume(self):
        while self.downloaded_bytes < self.total_bytes:
            start = self.downloaded_bytes
            end = min(start + self.chunk_size - 1, self.total_bytes - 1)
            
            chunk = self.download_chunk(start, end)
            self.data.extend(chunk)
            self.downloaded_bytes += len(chunk)
            
            progress = self.downloaded_bytes / self.total_bytes * 100
            yield progress

print("✅ ResumableDownloader ready!")

✅ ResumableDownloader ready!


In [5]:
print("🔄 Resumable Download Demo")
print("=" * 60)

downloader = ResumableDownloader(download_url, chunk_size=1024 * 1024)

print("\n1️⃣ Starting download (will 'fail' at 50%)...")
try:
    for progress in downloader.download(simulate_failure_at=0.5):
        bar = "█" * int(progress / 5) + "░" * (20 - int(progress / 5))
        print(f"   [{bar}] {progress:.0f}%", end="\r")
except ConnectionError as e:
    print(f"\n   ❌ {e}")

print(f"\n\n📊 Downloaded so far: {downloader.downloaded_bytes / 1024 / 1024:.1f}MB")
print(f"   Remaining: {(downloader.total_bytes - downloader.downloaded_bytes) / 1024 / 1024:.1f}MB")

print("\n2️⃣ Resuming download...")
for progress in downloader.resume():
    bar = "█" * int(progress / 5) + "░" * (20 - int(progress / 5))
    print(f"   [{bar}] {progress:.0f}%", end="\r")

print(f"\n\n✅ Download complete! Total: {len(downloader.data) / 1024 / 1024:.1f}MB")

🔄 Resumable Download Demo

1️⃣ Starting download (will 'fail' at 50%)...
   [██████████░░░░░░░░░░] 50%
   ❌ Simulated failure at 50%


📊 Downloaded so far: 5.0MB
   Remaining: 5.0MB

2️⃣ Resuming download...


   [████████████████████] 100%

✅ Download complete! Total: 10.0MB


## ⚡ Parallel Chunk Downloads

In [6]:
print("⚡ Parallel Chunk Downloads")
print("=" * 60)
print("""
Download multiple chunks simultaneously to maximize bandwidth.

SEQUENTIAL:
─────────────────────────────────────────────────────────────
Time: ──────────────────────────────────────────────────────>

      [Chunk 1]──────>[Chunk 2]──────>[Chunk 3]──────>[Chunk 4]
      Total: 4 units of time

PARALLEL (4 connections):
─────────────────────────────────────────────────────────────
Time: ──────────────────────────────────────────────────────>

      [Chunk 1]────>
      [Chunk 2]────>
      [Chunk 3]────>
      [Chunk 4]────>
      Total: 1 unit of time

4x speedup! (in ideal conditions)
""")

⚡ Parallel Chunk Downloads

Download multiple chunks simultaneously to maximize bandwidth.

SEQUENTIAL:
─────────────────────────────────────────────────────────────
Time: ──────────────────────────────────────────────────────>

      [Chunk 1]──────>[Chunk 2]──────>[Chunk 3]──────>[Chunk 4]
      Total: 4 units of time

PARALLEL (4 connections):
─────────────────────────────────────────────────────────────
Time: ──────────────────────────────────────────────────────>

      [Chunk 1]────>
      [Chunk 2]────>
      [Chunk 3]────>
      [Chunk 4]────>
      Total: 1 unit of time

4x speedup! (in ideal conditions)



In [7]:
def parallel_download(url: str, num_connections: int = 4) -> tuple:
    # Learn total size via a 1-byte range probe (works with GET-only URLs).
    probe = requests.get(url, headers={'Range': 'bytes=0-0'})
    content_range = probe.headers.get('Content-Range', '')
    total_size = int(content_range.split('/')[-1]) if '/' in content_range else 0
    
    chunk_size = total_size // num_connections
    ranges = []
    for i in range(num_connections):
        start = i * chunk_size
        end = start + chunk_size - 1 if i < num_connections - 1 else total_size - 1
        ranges.append((i, start, end))
    
    def download_range(args):
        idx, start, end = args
        response = requests.get(url, headers={'Range': f'bytes={start}-{end}'})
        return idx, response.content
    
    start_time = time.time()
    results = {}
    
    with ThreadPoolExecutor(max_workers=num_connections) as executor:
        futures = [executor.submit(download_range, r) for r in ranges]
        for future in as_completed(futures):
            idx, data = future.result()
            results[idx] = data
    
    elapsed = time.time() - start_time
    
    final_data = b''.join(results[i] for i in range(num_connections))
    return final_data, elapsed

def sequential_download(url: str) -> tuple:
    start_time = time.time()
    response = requests.get(url)
    elapsed = time.time() - start_time
    return response.content, elapsed

print("⚡ Sequential vs Parallel Download")
print("=" * 60)

print("\n1️⃣ Sequential download...")
seq_data, seq_time = sequential_download(download_url)
print(f"   Size: {len(seq_data) / 1024 / 1024:.1f}MB")
print(f"   Time: {seq_time:.2f}s")

print("\n2️⃣ Parallel download (4 connections)...")
par_data, par_time = parallel_download(download_url, num_connections=4)
print(f"   Size: {len(par_data) / 1024 / 1024:.1f}MB")
print(f"   Time: {par_time:.2f}s")

print(f"\n📊 Comparison:")
print(f"   Sequential: {seq_time:.2f}s")
print(f"   Parallel:   {par_time:.2f}s")
if par_time > 0:
    print(f"   Speedup:    {seq_time/par_time:.1f}x")

print(f"\n✅ Data integrity: {'Match!' if seq_data == par_data else 'MISMATCH!'}")

⚡ Sequential vs Parallel Download

1️⃣ Sequential download...
   Size: 10.0MB
   Time: 0.04s

2️⃣ Parallel download (4 connections)...
   Size: 10.0MB
   Time: 0.04s

📊 Comparison:
   Sequential: 0.04s
   Parallel:   0.04s
   Speedup:    1.0x

✅ Data integrity: Match!


## 🌐 CDN Basics

In [8]:
print("🌐 CDN Basics")
print("=" * 60)
print("""
CDNs cache content at edge locations worldwide.

WITHOUT CDN:
─────────────────────────────────────────────────────────────
    User in     200ms      Origin in
    Sydney  ─────────────> Virginia
    
    Every request travels around the world!

WITH CDN:
─────────────────────────────────────────────────────────────
                         ┌───────────────┐
                         │ Origin Server │
                         │  (Virginia)   │
                         └───────┬───────┘
                                 │ Cache miss
           ┌─────────────────────┼─────────────────────┐
           │                     │                     │
    ┌──────▼──────┐       ┌──────▼──────┐       ┌──────▼──────┐
    │ Edge: Tokyo │       │ Edge: London│       │ Edge: Sydney│
    └──────┬──────┘       └──────┬──────┘       └──────┬──────┘
           │ 10ms                │ 15ms                │ 5ms
    ┌──────▼──────┐       ┌──────▼──────┐       ┌──────▼──────┐
    │  User Japan │       │  User UK    │       │ User Sydney │
    └─────────────┘       └─────────────┘       └─────────────┘

BENEFITS:
• Latency: 200ms → 5-15ms
• Bandwidth: Origin doesn't serve every request
• Cost: Less egress from origin
• Availability: Edge can serve if origin is down
""")

🌐 CDN Basics

CDNs cache content at edge locations worldwide.

WITHOUT CDN:
─────────────────────────────────────────────────────────────
    User in     200ms      Origin in
    Sydney  ─────────────> Virginia

    Every request travels around the world!

WITH CDN:
─────────────────────────────────────────────────────────────
                         ┌───────────────┐
                         │ Origin Server │
                         │  (Virginia)   │
                         └───────┬───────┘
                                 │ Cache miss
           ┌─────────────────────┼─────────────────────┐
           │                     │                     │
    ┌──────▼──────┐       ┌──────▼──────┐       ┌──────▼──────┐
    │ Edge: Tokyo │       │ Edge: London│       │ Edge: Sydney│
    └──────┬──────┘       └──────┬──────┘       └──────┬──────┘
           │ 10ms                │ 15ms                │ 5ms
    ┌──────▼──────┐       ┌──────▼──────┐       ┌──────▼──────┐
    │  User Japan │   

## 🧪 Quick Quiz

1. **What HTTP status indicates a partial content response?**

2. **Why might parallel downloads NOT always be faster?**

3. **What's the benefit of CDN signed URLs vs origin signed URLs?**

In [9]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Partial content status:")
print("   - 206 Partial Content")
print("   - Includes Content-Range header")
print()
print("2. When parallel isn't faster:")
print("   - Total bandwidth is the limit, not connections")
print("   - Server may throttle per-IP")
print("   - Connection overhead for small files")
print()
print("3. CDN vs origin signed URLs:")
print("   - CDN validates at edge (faster)")
print("   - No call back to origin for validation")
print("   - Uses public/private key cryptography")

📝 Quiz Answers

1. Partial content status:
   - 206 Partial Content
   - Includes Content-Range header

2. When parallel isn't faster:
   - Total bandwidth is the limit, not connections
   - Server may throttle per-IP
   - Connection overhead for small files

3. CDN vs origin signed URLs:
   - CDN validates at edge (faster)
   - No call back to origin for validation
   - Uses public/private key cryptography


## 📚 Summary

### Key Takeaways

1. **Range requests** - Download parts of files (206 status)
2. **Resumable downloads** - Track bytes downloaded, resume from there
3. **Parallel downloads** - Multiple connections for throughput
4. **CDN caching** - Edge locations reduce latency
5. **Always support ranges** - Enable seeking and resuming

### Next Up

In **Notebook 6**, we'll learn about security and abuse prevention:
- Content validation
- Quarantine patterns
- Rate limiting uploads